# `defect_detect()`

`nematics3d.defect_detect()` locates nematic defects on the elementary square plaquettes of a three-dimensional director field $\mathbf{n}$. It returns lattice-index coordinates rather than physical coordinates.

## What the function detects

A nematic director is headless: $\mathbf{n}$ and $-\mathbf{n}$ represent the same local orientation. For each square plaquette, `defect_detect()` walks around its four corner directors, reverses signs when needed to keep successive directors aligned, and checks whether the orientation closes consistently. A failed closure marks a defect piercing that plaquette.

Each returned coordinate has one integer component and two half-integer components. The integer component identifies the axis normal to the pierced plaquette. For example:

| Coordinate pattern | Plaquette | Normal axis |
| --- | --- | --- |
| `(i, j + 0.5, k + 0.5)` | yz | x |
| `(i + 0.5, j, k + 0.5)` | xz | y |
| `(i + 0.5, j + 0.5, k)` | xy | z |

The result has shape `(number_of_defects, 3)`. When no defect is found, its shape is `(0, 3)`.

## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** The next cell imports `NumPy`, a timer, `Path`, and the public `nematics3d` package.

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import nematics3d as n3d

## Minimal example

The following `2 x 2 x 1` director field contains one defective xy plaquette. The final component is zero because the plaquette lies in the xy plane at `z=0`.

In [2]:
angles = np.array(
    [
        [0.0, 3 * np.pi / 4],
        [np.pi / 4, np.pi / 2],
    ]
)
n = np.zeros((2, 2, 1, 3))
n[..., 0] = np.cos(angles)[..., None]
n[..., 1] = np.sin(angles)[..., None]

defects = n3d.defect_detect(n, planes=(False, False, True))
print(defects)

[[0.5 0.5 0. ]]


A uniform director field has no winding and therefore returns an empty `(0, 3)` array.

In [3]:
uniform_n = np.zeros((3, 3, 2, 3))
uniform_n[..., 0] = 1.0
uniform_defects = n3d.defect_detect(uniform_n)
print(uniform_defects, uniform_defects.shape)

[] (0, 3)


## Arguments

```python
nematics3d.defect_detect(
    n_origin,
    threshold=0,
    is_boundary_periodic=0,
    planes=1,
    *,
    worker_count=None,
    is_input_validated=False,
)
```

| Argument | Meaning |
| --- | --- |
| `n_origin` | Real, finite director field with shape `(Nx, Ny, Nz, 3)` |
| `is_boundary_periodic` | One flag or three flags selecting periodic x, y, and z boundaries |
| `planes` | One flag or three flags selecting plaquettes normal to x, y, and z |
| `worker_count` | Explicit `NumExpr` thread count, or `None` for its current automatic setting |
| `is_input_validated` | Skip repeated director-field validation when the caller already guarantees the input contract |

The default `threshold=0` implements the current defect criterion and should normally be left unchanged. A scalar flag such as `planes=True` is broadcast to all three axes.

### Selecting plaquette orientations

The entries of `planes` describe plaquette **normals**, not the coordinate planes themselves:

- `(True, False, False)` checks yz plaquettes;
- `(False, True, False)` checks xz plaquettes;
- `(False, False, True)` checks xy plaquettes.

Selecting only the required orientations avoids unnecessary work. The minimal example uses only xy plaquettes.

### A defect crossing a periodic boundary

Without periodicity, no plaquette connects the last x slice to the first x slice. After x periodicity is enabled, that cross-boundary plaquette is included. Its center is reported at `Nx - 0.5`, which remains inside the original periodic cell.

In [4]:
periodic_angles = np.array(
    [
        [np.pi / 4, np.pi / 2],
        [0.0, 3 * np.pi / 4],
        [0.0, 3 * np.pi / 4],
    ]
)
periodic_n = np.zeros((3, 2, 1, 3))
periodic_n[..., 0] = np.cos(periodic_angles)[..., None]
periodic_n[..., 1] = np.sin(periodic_angles)[..., None]

without_periodicity = n3d.defect_detect(
    periodic_n,
    planes=(False, False, True),
)
with_x_periodicity = n3d.defect_detect(
    periodic_n,
    is_boundary_periodic=(True, False, False),
    planes=(False, False, True),
)

print("without x periodicity:")
print(without_periodicity)
print("with x periodicity:")
print(with_x_periodicity)

without x periodicity:
[[0.5 0.5 0. ]]
with x periodicity:
[[0.5 0.5 0. ]
 [2.5 0.5 0. ]]


The additional coordinate `[2.5, 0.5, 0.0]` is the defect crossing the x boundary. Here `Nx=3`, so its x coordinate is `3 - 0.5`. Periodicity is controlled independently along every axis.

### Starting from a $Q$-tensor field

Many workflows store $Q$ rather than $\mathbf{n}$. First call `q_diagonalize()` to obtain the principal director. That result already satisfies the shape, dtype, finiteness, and normalization requirements, so `is_input_validated=True` avoids scanning the complete field again.

The next example uses the $Q$-tensor field included with the repository.

In [5]:
def find_example_data():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        path = candidate / "example/data/Q_example_workflow.npy"
        if path.exists():
            return path
    raise FileNotFoundError(
        "Run this tutorial from inside the Nematics3D repository."
    )


Q_example = np.load(find_example_data())
diagonalized = n3d.q_diagonalize(Q_example)

start = perf_counter()
example_defects = n3d.defect_detect(
    diagonalized.n,
    is_input_validated=True,
)
elapsed = perf_counter() - start

print("director shape:", diagonalized.n.shape)
print("number of defects:", len(example_defects))
print(f"detection time: {elapsed:.4f} s")

director shape: (200, 100, 100, 3)
number of defects: 1270
detection time: 0.0304 s


### Validation and worker selection

Leave `is_input_validated=False` for user-provided arrays. The function then verifies the three-dimensional field shape, real dtype, and finite values. Set it to `True` only when an upstream operation already guarantees that contract; malformed trusted input is not required to produce a friendly validation error.

Most users should also leave `worker_count=None`. The `NumExpr` backend then uses its current thread configuration. An explicit positive count is useful for controlled benchmarks, but changing it temporarily modifies a process-wide `NumExpr` setting. Concurrent calls should configure `NumExpr` externally instead of requesting different counts per call.

## Details

For corners $\mathbf{a}\rightarrow\mathbf{b}\rightarrow\mathbf{c}\rightarrow\mathbf{d}$, the nematic sign alignment can be reduced to the sign product of three successive edge dot products. The aligned closure quantity is

$$
(\mathbf{a}\mathbin{\cdot}\mathbf{d})
\,\operatorname{sgn}(\mathbf{a}\mathbin{\cdot}\mathbf{b})
\,\operatorname{sgn}(\mathbf{b}\mathbin{\cdot}\mathbf{c})
\,\operatorname{sgn}(\mathbf{c}\mathbin{\cdot}\mathbf{d}),
$$

where a zero edge dot product uses the positive sign convention. The default test marks a defect when this quantity is negative. `NumExpr` evaluates the complete predicate in fused multithreaded passes, avoiding the large stacked and sign-aligned director arrays required by a direct implementation.

The magnitude of each nonzero director does not affect this sign-based criterion, so the safe input path validates the field without renormalizing it.

## Possible issues

### The input shape is rejected

The function requires shape `(Nx, Ny, Nz, 3)`. A two-dimensional field must still include a singleton spatial axis, for example `(Nx, Ny, 1, 3)`. Use `planes=(False, False, True)` for xy plaquettes in that representation.

### A periodic defect is missing

Check that `is_boundary_periodic` enables the spatial axis crossed by the plaquette. This is independent of `planes`, which selects the plaquette's normal axis.

### Zero directors are present

A zero director has no physical orientation. Although the array can be processed, detections involving undefined directors should not be interpreted physically. Mask or repair such regions before defect analysis.

### Output coordinates are not physical positions

The returned values are lattice indices at plaquette centers. Apply the field's grid transform when real-space positions are required. Higher-level objects such as `QFieldObject` perform that conversion for their stored results.

## Where `defect_detect()` is used

- `QFieldObject.act_defect_detect()` detects defects from the object's prepared director field and converts their lattice indices into real-space grid coordinates.
- `QPlane` uses a singleton third spatial axis and selects only xy plaquettes when detecting defects on a sampled plane.

Call the low-level function directly when the director array, selected plaquette orientations, or periodicity need to be controlled explicitly.

## Useful Links

### Referenced in this tutorial

- [`q_diagonalize()`](q_diagonalize.ipynb) — converts a $Q$-tensor field into $S$ and the principal director $\mathbf{n}$.
- [`get_q()`](../field/get_q.ipynb) — constructs $Q$ from scalar order and director data.
- [`QFieldObject.act_defect_detect()`](../classes/QFieldObject/act_defect_detect.ipynb) — performs defect detection through the higher-level field object.